## 📖 How to use this notebook

- **Open it in Google Colab** and choose `File → Save a copy in Drive` first, so your work is kept.
- **Run a cell** by clicking on it and pressing **Shift + Enter** (or the ▶ button on its left). **Run the cells in order, top to bottom** — later cells use results from earlier ones.
- **✏️ TODO** cells contain a blank written as `___`. Replace the `___` (underscores included) with your answer, then run the cell. Each blank is one small thing — usually a single word or number.
- **💡 Hint** lines in the TODO cell tell you what to type.
- **✅ Stuck?** Right below every TODO there is a hidden answer. Click it to reveal. Using it is completely fine — the goal is to understand what the code does, not to type it from memory.
- **⭐ Optional** sections are extras for if you finish early or want to explore after the workshop. You can skip them; nothing later in the notebook depends on them.
- **Lines starting with `#` are comments** written for you to read. Python ignores them.
- **If something goes wrong:** `Runtime → Restart session`, then run the cells from the top again. Or raise your hand — that's what the TAs are for.

No coding background is needed.

# Session 2 — Machine Learning & RNA Data Analysis

**Today's goal:** take a set of RNA sequences, turn them into numbers a computer can learn from, and train and evaluate a classifier that predicts whether an RNA is **coding** (mRNA) or **non-coding**.

| Part | Topic | Time |
|---|---|---|
| Setup | Install and import the tools | ~3 min |
| 1 | Classification, regression or clustering? | ~7 min |
| 2 | Clean & prepare the RNA dataset | ~15 min |
| 3 | Train classifiers | ~10 min |
| 4 | Cross-validation (⭐ + optional hyperparameter tuning) | ~7 min |
| 5 | Evaluate the models | ~12 min |
| 6 | Interpret what the model learned | ~5 min |
| ⭐ | Optional bonus: unsupervised clustering of the real data | ~10 min |

### 📚 A few words you'll see today
- **Feature** — a number describing each sample (e.g. sequence length). The model learns from these.
- **Label** — the answer we want to predict (here: coding or non-coding).
- **Training set / test set** — we learn from one part of the data and check on a part the model has *never seen*, like an exam with new questions.
- **Overfitting** — memorising the training data instead of learning a general rule. Great on training data, worse on new data.

---
## Setup — run this first ⏱ ~3 min

Installs and loads everything for today: **pandas** (tables), **NumPy** (numbers), **matplotlib** (plots), and **scikit-learn** (machine learning). Wait for `Setup complete.`

In [ ]:
!pip install -q biopython requests pandas scikit-learn matplotlib

import requests
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split, cross_val_score, GridSearchCV, KFold
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    confusion_matrix, ConfusionMatrixDisplay, roc_curve, roc_auc_score,
)

pd.set_option("display.max_colwidth", 80)
np.random.seed(42)
print("Setup complete.")

---
## Part 1 — Classification, Regression, or Clustering? ⏱ ~7 min
*(companion to slide: "Classification, Regression, or Clustering?")*

Before touching real RNA data, let's build intuition with three tiny made-up datasets. Machine-learning problems come in three main flavours:
- **Classification** — predict a *category* (e.g. coding / non-coding) from labelled examples
- **Regression** — predict a *number* (e.g. an expression level) from labelled examples
- **Clustering** — no labels at all; find natural *groups* in the data

👉 Run the three cells below and decide which flavour each dataset is.

In [ ]:
# Toy dataset A
np.random.seed(0)
lengthA = np.random.normal(500, 100, 40)
labelA = np.where(lengthA > 500, "coding", "non-coding")

plt.figure(figsize=(5, 3))
for lbl, color in [("coding", "teal"), ("non-coding", "darkorange")]:
    mask = labelA == lbl
    plt.scatter(lengthA[mask], np.zeros(mask.sum()), label=lbl, alpha=0.7, color=color)
plt.yticks([])
plt.xlabel("Sequence length")
plt.title("Dataset A — has labels")
plt.legend()
plt.show()

print("Dataset A: what kind of problem is this?")

In [ ]:
# Toy dataset B
gc_content = np.random.uniform(30, 70, 40)
expression_level = 2.5 * gc_content + np.random.normal(0, 15, 40)

plt.figure(figsize=(5, 3))
plt.scatter(gc_content, expression_level, alpha=0.7, color="teal")
plt.xlabel("GC content (%)")
plt.ylabel("Expression level")
plt.title("Dataset B — continuous target")
plt.show()

print("Dataset B: what kind of problem is this?")

In [ ]:
# Toy dataset C
from numpy.random import multivariate_normal

groupA = multivariate_normal([30, 40], [[15, 0], [0, 15]], 20)
groupB = multivariate_normal([60, 70], [[15, 0], [0, 15]], 20)
unlabeled = np.vstack([groupA, groupB])
np.random.shuffle(unlabeled)

plt.figure(figsize=(5, 3))
plt.scatter(unlabeled[:, 0], unlabeled[:, 1], alpha=0.7, color="gray")
plt.xlabel("Feature 1")
plt.ylabel("Feature 2")
plt.title("Dataset C — no labels at all")
plt.show()

print("Dataset C: what kind of problem is this?")

🤔 **Your answers:** decide for A, B and C before revealing.

<details>
<summary>✅ <b>Click to reveal</b></summary>

- **A — classification:** each point has a category label (coding / non-coding).
- **B — regression:** the target (expression level) is a continuous number.
- **C — clustering:** there are no labels; we can only look for groups.

</details>

### ⭐ Optional — try clustering dataset C yourself

**K-Means** is the classic clustering algorithm: you tell it how many groups to look for, and it finds the best group centres. Nothing later depends on this cell.

In [ ]:
from sklearn.cluster import KMeans

# ✏️ TODO: How many clusters should K-Means look for? Look at the plot of dataset C.
# 💡 Hint: there are 2 visible groups, so fill in  2
kmeans = KMeans(n_clusters=___, random_state=42, n_init=10)
cluster_labels = kmeans.fit_predict(unlabeled)

plt.figure(figsize=(5, 3))
plt.scatter(unlabeled[:, 0], unlabeled[:, 1], c=cluster_labels, cmap="viridis", alpha=0.7)
plt.scatter(kmeans.cluster_centers_[:, 0], kmeans.cluster_centers_[:, 1], c="red", marker="X", s=150, label="centers")
plt.xlabel("Feature 1")
plt.ylabel("Feature 2")
plt.title("K-Means recovered the groups, with no labels")
plt.legend()
plt.show()

<details>
<summary>✅ <b>Stuck? Click here to reveal the answer</b></summary>

```python
kmeans = KMeans(n_clusters=2, random_state=42, n_init=10)
```

</details>

---
## Part 2 — Clean & Prepare Your RNA Dataset ⏱ ~15 min
*(companion to slide: "Clean & Prepare Your RNA Dataset")*

**Why mRNA vs. lncRNA, not mRNA vs. miRNA?** miRNAs are tiny (~60–110 nt) compared to mRNAs (300–2000+ nt) — `length` alone would perfectly separate them, and any model would trivially score 1.0 on everything. That's not a good model, it's an easy dataset.

**lncRNAs** (long non-coding RNAs) are a much more realistic challenge: they're similar in *length* to mRNAs, but typically lack long open reading frames. This is a classic bioinformatics problem (tools like CPAT and CPC2 exist specifically for it) — expect real overlap between the classes, and metrics that aren't a perfect 1.0.

### 2.1 Download the data
This cell downloads non-coding sequences from RNAcentral and coding mRNAs from NCBI, using the API skills from Session 1. 👉 **Just run it** (it can take ~30 seconds). If the internet is slow or a database is down, it automatically switches to a built-in backup dataset, so you can always continue.

In [ ]:
def fetch_noncoding(n=100):
    """Non-coding lncRNA sequences from RNAcentral — chosen to overlap in length with mRNA."""
    url = "https://rnacentral.org/api/v1/rna"
    params = {"query": "TAXONOMY:9606 AND rna_type:lncRNA", "format": "json", "page_size": n}
    resp = requests.get(url, params=params, timeout=120)
    resp.raise_for_status()
    results = resp.json().get("results", [])
    rows = []
    for r in results:
        seq = (r.get("sequence") or "")
        if seq:
            rows.append({"sequence": seq, "label": "non-coding"})
    return rows

def fetch_coding(n=100):
    """Coding mRNA sequences from NCBI (RefSeq), length range overlapping lncRNA."""
    base = "https://eutils.ncbi.nlm.nih.gov/entrez/eutils"
    search = requests.get(f"{base}/esearch.fcgi", params={
        "db": "nucleotide",
        "term": "Homo sapiens[Organism] AND biomol_mrna[PROP] AND refseq[filter] AND 200:3000[SLEN]",
        "retmax": n, "retmode": "json",
    }, timeout=15).json()
    ids = search["esearchresult"]["idlist"]
    if not ids:
        return []
    fetch = requests.get(f"{base}/efetch.fcgi", params={
        "db": "nucleotide", "id": ",".join(ids), "rettype": "fasta", "retmode": "text",
    }, timeout=30)
    rows, seq = [], ""
    for line in fetch.text.splitlines():
        if line.startswith(">"):
            if seq:
                rows.append({"sequence": seq, "label": "coding"})
            seq = ""
        else:
            seq += line.strip()
    if seq:
        rows.append({"sequence": seq, "label": "coding"})
    return rows

def synthetic_fallback(n=100):
    """
    Guaranteed-to-work backup if the APIs are unreachable during the workshop.
    Deliberately overlapping distributions — an easy 1.0-accuracy dataset teaches nothing.
    """
    rows = []
    bases = list("ACGU")
    # non-coding (lncRNA-like): long, GC-content close to coding, short/sparse ORFs
    for _ in range(n):
        length = int(np.random.normal(700, 400))
        length = max(length, 150)
        seq = "".join(np.random.choice(bases, size=length, p=[0.26, 0.24, 0.24, 0.26]))
        rows.append({"sequence": seq, "label": "non-coding"})
    # coding (mRNA-like): similar length range, slightly different GC, longer ORFs on average
    for _ in range(n):
        length = int(np.random.normal(900, 450))
        length = max(length, 150)
        seq = "".join(np.random.choice(bases, size=length, p=[0.24, 0.26, 0.26, 0.24]))
        rows.append({"sequence": seq, "label": "coding"})
    return rows

try:
    rows = fetch_noncoding(100) + fetch_coding(100)
    if len(rows) < 60:
        raise ValueError("Too few records returned")
    print(f"Fetched {len(rows)} real sequences from RNAcentral + NCBI")
except Exception as e:
    print(f"Live fetch failed ({e}) — using synthetic fallback dataset instead")
    rows = synthetic_fallback(100)

raw_df = pd.DataFrame(rows)
raw_df.head()

### 2.2 Sanity-check the raw data
Always look at your data before modelling. Three quick questions: how big is it, are the classes balanced, and is anything missing?

In [ ]:
print("Rows, columns:", raw_df.shape)

# ✏️ TODO: count how many sequences carry each label.
# 💡 Hint: the column with the labels is called "label" (in quotes)
print(raw_df[___].value_counts())

print("\nMissing values per column:")
print(raw_df.isna().sum())

<details>
<summary>✅ <b>Stuck? Click here to reveal the answer</b></summary>

```python
print(raw_df["label"].value_counts())
```

</details>

### 2.3 Clean it
Remove duplicate sequences and rows with missing values, then write every sequence in the RNA alphabet (uppercase, `U` instead of `T`).

In [ ]:
# ✏️ TODO: remove duplicate sequences. Fill in the name of the column to check for duplicates.
# 💡 Hint: "sequence"
clean_df = raw_df.drop_duplicates(subset=___).dropna(subset=["sequence", "label"]).reset_index(drop=True)

clean_df["sequence"] = clean_df["sequence"].str.upper().str.replace("T", "U")   # RNA alphabet
print(f"{len(raw_df)} -> {len(clean_df)} rows after cleaning")

<details>
<summary>✅ <b>Stuck? Click here to reveal the answer</b></summary>

```python
clean_df = raw_df.drop_duplicates(subset="sequence").dropna(subset=["sequence", "label"]).reset_index(drop=True)
```

</details>

### 2.4 Feature engineering — turn sequences into numbers
A model can't read `AUGGC…` directly, so we describe each sequence with three numbers:
- **length** — number of nucleotides
- **gc_content** — % of G and C bases
- **orf_length** — length of the longest **open reading frame** (an `AUG` start codon followed by codons up to a stop codon). Coding RNAs need a long ORF to make a protein, so this should be informative.

The two helper functions are written for you. Your job is to apply the ORF function to every sequence.

In [ ]:
# --- feature engineering ---
def gc_content(seq):
    if len(seq) == 0:
        return 0.0
    return (seq.count("G") + seq.count("C")) / len(seq) * 100

def longest_orf_length(seq):
    '''Very simplified ORF finder: longest AUG...stop run in the sequence.'''
    stops = {"UAA", "UAG", "UGA"}
    best = 0
    for frame in range(3):
        i = frame
        while i < len(seq) - 2:
            if seq[i:i+3] == "AUG":
                j = i + 3
                while j < len(seq) - 2 and seq[j:j+3] not in stops:
                    j += 3
                best = max(best, j - i)
                i = j
            else:
                i += 3
    return best

clean_df["length"] = clean_df["sequence"].str.len()
clean_df["gc_content"] = clean_df["sequence"].apply(gc_content)

# ✏️ TODO: apply the ORF function to every sequence, just like the line above does for GC content.
# 💡 Hint: the function is called  longest_orf_length   (no quotes, no brackets)
clean_df["orf_length"] = clean_df["sequence"].apply(___)

clean_df[["sequence", "label", "length", "gc_content", "orf_length"]].head()

<details>
<summary>✅ <b>Stuck? Click here to reveal the answer</b></summary>

```python
clean_df["orf_length"] = clean_df["sequence"].apply(longest_orf_length)
```

</details>

Let's look at how the two classes compare on each feature. Where the histograms overlap, that feature alone can't tell the classes apart.

In [ ]:
# --- quick sanity check: is this dataset actually harder than length alone would suggest? ---
fig, axes = plt.subplots(1, 3, figsize=(13, 3.5))
for ax, col in zip(axes, ["length", "gc_content", "orf_length"]):
    for lbl, color in [("coding", "#0E7C7B"), ("non-coding", "#E8963A")]:
        ax.hist(clean_df.loc[clean_df["label"] == lbl, col], bins=20, alpha=0.6, label=lbl, color=color)
    ax.set_title(col)
    ax.legend()
plt.tight_layout()
plt.show()
print("Notice the overlap — unlike mRNA vs. miRNA, these classes are not trivially separable.")

### 2.5 Encode the label
Models work with numbers, not words, so we turn `"coding"` / `"non-coding"` into `0` / `1`. `LabelEncoder` assigns numbers in alphabetical order.

In [ ]:
le = LabelEncoder()

# ✏️ TODO: give the encoder the label column.
# 💡 Hint: clean_df["label"]
clean_df["label_encoded"] = le.fit_transform(___)
print(dict(zip(le.classes_, le.transform(le.classes_))))

<details>
<summary>✅ <b>Stuck? Click here to reveal the answer</b></summary>

```python
clean_df["label_encoded"] = le.fit_transform(clean_df["label"])
```

</details>

### 2.6 Scale the features
Length is in the hundreds or thousands, GC content is between 0 and 100. **Scaling** puts every feature on the same footing (mean 0, standard deviation 1) so no feature dominates just because its numbers are bigger. 👉 Just run it.

In [ ]:
feature_cols = ["length", "gc_content", "orf_length"]
scaler = StandardScaler()

X_scaled = scaler.fit_transform(clean_df[feature_cols])
X_scaled_df = pd.DataFrame(X_scaled, columns=[c + "_scaled" for c in feature_cols])
clean_df = pd.concat([clean_df, X_scaled_df], axis=1)
clean_df.head()

### 2.7 Split into training and test sets
We hide 20% of the data as a **test set** that the model never sees during training. `stratify=y` keeps the coding/non-coding balance the same in both parts.

In [ ]:
X = clean_df[[c + "_scaled" for c in feature_cols]]   # the features
y = clean_df["label_encoded"]                           # the answers

# ✏️ TODO: what fraction of the data should go into the test set?
# 💡 Hint: 20% is written as  0.2
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=___, random_state=42, stratify=y)

print(f"Train: {X_train.shape[0]} rows | Test: {X_test.shape[0]} rows")
print("Class balance (train):", y_train.value_counts().to_dict())

<details>
<summary>✅ <b>Stuck? Click here to reveal the answer</b></summary>

```python
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)
```

</details>

---
## Part 3 — Train the Classifier ⏱ ~10 min
*(companion to slide: "Train the Classifier")*

Every scikit-learn model follows the same three steps:
1. **create** it — `model = SomeModel()`
2. **fit** it on the training data — `model.fit(X_train, y_train)`
3. **predict** on new data — `model.predict(X_test)`

First, a single **decision tree**: a flowchart of yes/no questions ("is orf_length > 300?"). Watch how its training and test accuracy compare.

In [ ]:
tree = DecisionTreeClassifier(random_state=42)

# ✏️ TODO: train the tree on the training data (features first, then answers).
# 💡 Hint: X_train, y_train
tree.fit(___, ___)

train_acc_tree = accuracy_score(y_train, tree.predict(X_train))
test_acc_tree = accuracy_score(y_test, tree.predict(X_test))
print(f"Single Decision Tree — train accuracy: {train_acc_tree:.3f}   test accuracy: {test_acc_tree:.3f}")
print("A big gap between these two numbers is a sign of overfitting.")

<details>
<summary>✅ <b>Stuck? Click here to reveal the answer</b></summary>

```python
tree.fit(X_train, y_train)
```

</details>

Now two more models with the same create → fit → predict pattern:
- **Logistic regression** — a simple, classic model that draws a straight boundary between the classes
- **Random forest** — many decision trees, each trained on a random sample of the data, voting together. Averaging many trees usually reduces overfitting.

In [ ]:
log_reg = LogisticRegression(random_state=42)
log_reg.fit(X_train, y_train)
y_pred_lr = log_reg.predict(X_test)

# ✏️ TODO: how many trees should the forest have?
# 💡 Hint: use 200
rf = RandomForestClassifier(n_estimators=___, random_state=42)
rf.fit(X_train, y_train)
y_pred_rf = rf.predict(X_test)

train_acc_rf = accuracy_score(y_train, rf.predict(X_train))
test_acc_rf = accuracy_score(y_test, y_pred_rf)
print(f"Logistic Regression — test accuracy: {accuracy_score(y_test, y_pred_lr):.3f}")
print(f"Random Forest       — train accuracy: {train_acc_rf:.3f}   test accuracy: {test_acc_rf:.3f}")
print("Compare the forest's train/test gap to the single tree above.")

<details>
<summary>✅ <b>Stuck? Click here to reveal the answer</b></summary>

```python
rf = RandomForestClassifier(n_estimators=200, random_state=42)
```

</details>

---
## Part 4 — Cross-Validation ⏱ ~7 min
*(companion to slide: "Cross-Validation & GridSearchCV")*

A single train/test split can be lucky or unlucky. **5-fold cross-validation** splits the data into 5 parts, trains 5 times — each time testing on a different part — and gives you 5 scores. The spread of those scores tells you how much to trust a single number.

In [ ]:
# ✏️ TODO: how many folds?
# 💡 Hint: 5-fold cross-validation means  5
cv_scores = cross_val_score(RandomForestClassifier(n_estimators=200, random_state=42), X, y, cv=___, scoring="accuracy")

print("5-fold CV accuracy scores:", np.round(cv_scores, 3))
print(f"Mean: {cv_scores.mean():.3f}   Std: {cv_scores.std():.3f}")
print(f"\nSingle train/test split gave: {test_acc_rf:.3f}")
print("If these numbers disagree a lot, the single split was misleading.")

<details>
<summary>✅ <b>Stuck? Click here to reveal the answer</b></summary>

```python
cv_scores = cross_val_score(RandomForestClassifier(n_estimators=200, random_state=42), X, y, cv=5, scoring="accuracy")
```

</details>

### ⭐ Optional — tune the hyperparameters with GridSearchCV
**Hyperparameters** are settings you choose *before* training (e.g. how many trees, how deep each tree can grow). `GridSearchCV` tries every combination in a grid, cross-validating each one, and keeps the best.

> **Optional:** this tries 12 combinations × 5 folds = 60 trainings, so it takes ~30–60 seconds. If you skip it, Part 5 automatically uses the default random forest instead. 👉 Just run it.

In [ ]:
param_grid = {
    "max_depth": [3, 5, 10, None],
    "n_estimators": [50, 100, 200],
}

grid_search = GridSearchCV(RandomForestClassifier(random_state=42), param_grid, cv=5, scoring="accuracy", n_jobs=-1)
grid_search.fit(X_train, y_train)

print("Best hyperparameters found:", grid_search.best_params_)
print(f"Best CV accuracy: {grid_search.best_score_:.3f}")

best_rf = grid_search.best_estimator_
best_name = "Random Forest (tuned)"
tuned_test_acc = accuracy_score(y_test, best_rf.predict(X_test))
print(f"Tuned model — held-out test accuracy: {tuned_test_acc:.3f}")

🗣️ **Discussion:** did tuning actually help, or was the default `n_estimators=200` already close to optimal? Not every model needs heavy tuning — but you only know that *after* checking.

---
## Part 5 — Evaluate Your Models ⏱ ~12 min
*(companion to slide: "Evaluate Your Models")*

Accuracy alone can hide problems. Four standard metrics:
- **Accuracy** — fraction of all predictions that were correct
- **Precision** — of the samples predicted *positive*, how many really were?
- **Recall** — of the truly *positive* samples, how many did we find?
- **F1** — a single score that balances precision and recall

⚠️ **Which class is "positive"?** `LabelEncoder` numbered the classes alphabetically, so **coding = 0** and **non-coding = 1**. Precision and recall below therefore describe how well each model finds **non-coding** RNAs.

The first cell makes sure we have a "best" random forest, even if you skipped the optional tuning.

In [ ]:
# If the optional GridSearchCV step was skipped, use the default random forest as the "best" model
if "best_rf" not in globals():
    best_rf = rf
    best_name = "Random Forest (default — tuning skipped)"
print("Model used as best_rf:", best_name)

In [ ]:
def evaluate(name, y_true, y_pred):
    return {
        "model": name,
        "accuracy": accuracy_score(y_true, y_pred),
        "precision": precision_score(y_true, y_pred),
        "recall": recall_score(y_true, y_pred),
        # ✏️ TODO: fill in the function that computes the F1 score (same pattern as the lines above).
        # 💡 Hint: f1_score
        "f1": ___(y_true, y_pred),
    }

y_pred_best_rf = best_rf.predict(X_test)

model_rows = [evaluate("Logistic Regression", y_test, y_pred_lr),
              evaluate("Random Forest (default)", y_test, y_pred_rf)]
if best_rf is not rf:
    model_rows.append(evaluate(best_name, y_test, y_pred_best_rf))

results = pd.DataFrame(model_rows)
results

<details>
<summary>✅ <b>Stuck? Click here to reveal the answer</b></summary>

```python
"f1": f1_score(y_true, y_pred),
```

</details>

A **confusion matrix** shows exactly where each model is right and wrong: rows are the true class, columns the predicted class. The diagonal is correct predictions; everything off the diagonal is a mistake.

In [ ]:
models_to_plot = [("Logistic Regression", y_pred_lr), ("Random Forest (default)", y_pred_rf)]
if best_rf is not rf:
    models_to_plot.append((best_name, y_pred_best_rf))

fig, axes = plt.subplots(1, len(models_to_plot), figsize=(4.7 * len(models_to_plot), 4))
for ax, (name, y_pred) in zip(axes, models_to_plot):
    cm = confusion_matrix(y_test, y_pred)
    disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=le.classes_)
    disp.plot(ax=ax, colorbar=False)
    ax.set_title(name, fontsize=10)
plt.tight_layout()
plt.show()

### ROC curve & AUC
Models actually output a *probability*, and we turn it into a class with a cut-off (normally 0.5). The **ROC curve** shows the trade-off between catching positives (true positive rate) and false alarms (false positive rate) at every possible cut-off. **AUC** (area under the curve) summarises it in one number: 1.0 = perfect, 0.5 = random guessing.

In [ ]:
plt.figure(figsize=(6, 5))
for name, model in [("Logistic Regression", log_reg), (best_name, best_rf)]:
    y_proba = model.predict_proba(X_test)[:, 1]      # predicted probability of class 1
    fpr, tpr, _ = roc_curve(y_test, y_proba)
    # ✏️ TODO: compute the AUC from the true answers and the predicted probabilities.
    # 💡 Hint: the probabilities are stored in  y_proba
    auc = roc_auc_score(y_test, ___)
    plt.plot(fpr, tpr, label=f"{name} (AUC={auc:.3f})")

plt.plot([0, 1], [0, 1], linestyle="--", color="gray", label="Random guess")
plt.xlabel("False Positive Rate")
plt.ylabel("True Positive Rate")
plt.title("ROC Curve — Coding vs. Non-Coding RNA")
plt.legend()
plt.show()

<details>
<summary>✅ <b>Stuck? Click here to reveal the answer</b></summary>

```python
auc = roc_auc_score(y_test, y_proba)
```

</details>

🗣️ **Discussion:** which model would you trust more for this task, and why? Think about what a false negative (missing a coding RNA) vs. a false positive (calling a non-coding RNA coding) would mean for downstream research — and whether the ROC curve changes your answer.

---
## Part 6 — Interpret What the Model Learned ⏱ ~5 min
*(companion to slide: "Interpret What the Model Learned")*

A random forest can report **feature importance**: how much each feature helped its trees make good decisions. Higher = the model relied on it more.

In [ ]:
# ✏️ TODO: label each importance with its feature name.
# 💡 Hint: the list of feature names is called  feature_cols
importances = pd.Series(best_rf.feature_importances_, index=___).sort_values(ascending=False)

plt.figure(figsize=(6, 4))
importances.plot(kind="barh", color="#0E7C7B")
plt.gca().invert_yaxis()
plt.xlabel("Importance")
plt.title(f"Feature importance — {best_name}")
plt.show()

importances

<details>
<summary>✅ <b>Stuck? Click here to reveal the answer</b></summary>

```python
importances = pd.Series(best_rf.feature_importances_, index=feature_cols).sort_values(ascending=False)
```

</details>

🗣️ **Discussion:** look back at the histograms in Part 2.4. Does the feature with the most overlap between the classes get the lowest importance here? Discuss with your neighbour whether the model's reasoning lines up with what we know about coding-sequence biology (long ORFs, subtle GC bias) — and why this task is harder than getting a perfect 1.0.

---
## Save Your Results (for the Homework)

Save the cleaned dataset and the model metrics — you'll extend them after class. In Colab this also downloads both files to your computer (your browser may ask for permission to download multiple files).

In [ ]:
clean_df.to_csv("session2_rna_dataset_clean.csv", index=False)
results.to_csv("session2_model_results.csv", index=False)

try:
    from google.colab import files
    files.download("session2_rna_dataset_clean.csv")
    files.download("session2_model_results.csv")
except ImportError:
    print("Not running in Colab — the files were saved in the current folder.")

print("Saved. See you for the homework — and Session 3!")

---
## ⭐ Optional Bonus — Cluster the Real Dataset, Unsupervised ⏱ ~10 min

> **Optional:** for if you finish early or want to explore after the workshop.

In Part 1 we ran K-Means on a *toy* dataset. Now try it on the real data: pretend the `label` column doesn't exist, and see whether clustering on our three scaled features (`length`, `gc_content`, `orf_length`) rediscovers the coding vs. non-coding split on its own.

In [ ]:
from sklearn.cluster import KMeans
from sklearn.metrics import adjusted_rand_score

# ✏️ TODO: same K-Means pattern as Part 1 — look for 2 clusters, this time on the real features X.
# 💡 Hint: n_clusters should be 2, and the data to cluster is  X
kmeans_real = KMeans(n_clusters=___, random_state=42, n_init=10)
cluster_labels_real = kmeans_real.fit_predict(___)

print("Cluster sizes:", pd.Series(cluster_labels_real).value_counts().to_dict())

<details>
<summary>✅ <b>Stuck? Click here to reveal the answer</b></summary>

```python
kmeans_real = KMeans(n_clusters=2, random_state=42, n_init=10)
cluster_labels_real = kmeans_real.fit_predict(X)
```

</details>

Now compare the clusters to the true labels, which K-Means never saw. The **Adjusted Rand Index (ARI)** scores the agreement: 1.0 = identical groupings, 0 = no better than random.

In [ ]:
comparison_table = pd.crosstab(
    clean_df["label"], cluster_labels_real,
    rownames=["True label"], colnames=["K-Means cluster"]
)
print(comparison_table)

# ✏️ TODO: compare the true labels y with the cluster labels.
# 💡 Hint: the cluster labels are stored in  cluster_labels_real
ari = adjusted_rand_score(y, ___)
print(f"\nAdjusted Rand Index: {ari:.3f}  (1.0 = perfect agreement with true labels, 0.0 = random)")

<details>
<summary>✅ <b>Stuck? Click here to reveal the answer</b></summary>

```python
ari = adjusted_rand_score(y, cluster_labels_real)
```

</details>

🗣️ **Discussion:** did K-Means land close to the true coding/non-coding split, or did it group the data along some other axis of variation? What does that tell you about these three features being more useful for classification (which uses the labels) than for clustering (which doesn't)?

---
## Wrap-Up

**What you built today:**
- Intuition for classification vs. regression vs. clustering
- A cleaned, labelled, scaled coding vs. non-coding RNA dataset with three engineered features
- A single decision tree (watch for overfitting!), logistic regression and a random forest
- Cross-validation (⭐ and optional GridSearchCV tuning)
- A full evaluation: accuracy / precision / recall / F1, confusion matrices and ROC/AUC
- A biological interpretation of which features mattered
- ⭐ *(optional)* an unsupervised K-Means check against the true labels

**Homework:** extend this classifier with new features and bring your results to Session 3, where we'll compare it against a pretrained RNA foundation model.